In [ ]:
!pip install numpy pandas scipy rasterio matplotlib tqdm

In [ ]:
# import the packages
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from tqdm import tqdm

In [ ]:
# Function to read ASCII LiDAR file and extract vegetation points
def read_vegetation_points(file_path):
    """
    Reads a space-separated ASCII file containing LiDAR points
    and extracts vegetation points.

    Parameters:
        file_path (str): Path to the ASCII .txt file.

    Returns:
        pd.DataFrame: Dataframe containing vegetation points (X, Y, Z).
    """
    # Read file with space-separated values
    df = pd.read_csv(file_path, sep='\s+', names=['X', 'Y', 'Z', 'Class'], engine='python')

    # Print unique class values for debugging
    print("\nUnique classification values in dataset:", df['Class'].unique())

    # Filter vegetation points (assuming vegetation class is 1)
    vegetation_points = df[df['Class'] == 1].reset_index(drop=True)

    print(f"\nTotal vegetation points extracted: {len(vegetation_points)}")

    return vegetation_points

In [ ]:
# Load the DTM from the previously saved DTM GeoTIFF
def load_dtm_from_tif(dtm_file_path):
    """
    Loads a DTM GeoTIFF file and retrieves spatial information.

    Parameters:
        dtm_file_path (str): Path to the DTM file.

    Returns:
        np.array: DTM grid.
        float: Minimum X coordinate.
        float: Maximum Y coordinate.
        float: Resolution of the DTM grid.
    """
    with rasterio.open(dtm_file_path) as dataset:
        dtm_array = dataset.read(1)
        transform = dataset.transform

        # Extract spatial information
        x_min = transform.c
        y_max = transform.f
        resolution = transform.a

    print(f"✅ DTM loaded successfully with resolution: {resolution} meters")
    return dtm_array, x_min, y_max, resolution


In [ ]:
# Normalize vegetation points using the DTM grid
def normalize_vegetation(vegetation_df, dtm, x_min, y_max, resolution):
    """
    Normalizes vegetation heights by subtracting terrain elevation from the DTM.

    Parameters:
        vegetation_df (pd.DataFrame): DataFrame containing vegetation points (X, Y, Z).
        dtm (np.array): DTM grid data.
        x_min (float): Minimum X coordinate of the grid.
        y_max (float): Maximum Y coordinate of the grid.
        resolution (float): Grid resolution (e.g., 10m).

    Returns:
        pd.DataFrame: DataFrame with normalized vegetation heights.
    """
    # Get grid dimensions
    dtm_height, dtm_width = dtm.shape

    # Compute row, col indices for each vegetation point
    col_indices = ((vegetation_df['X'] - x_min) / resolution).astype(int)
    row_indices = ((y_max - vegetation_df['Y']) / resolution).astype(int)

    # Ensure indices are within bounds
    row_indices = np.clip(row_indices, 0, dtm_height - 1)
    col_indices = np.clip(col_indices, 0, dtm_width - 1)

    # Extract DTM height values at vegetation locations
    terrain_heights = dtm[row_indices, col_indices]

    # Compute normalized heights (Vegetation Height - Terrain Height)
    vegetation_df['Z_normalized'] = vegetation_df['Z'] - terrain_heights

    return vegetation_df

In [ ]:

# Save normalized vegetation points to a GeoTIFF file
def save_normalized_vegetation_raster(vegetation_df, x_min, y_max, resolution, output_file):
    """
    Converts normalized vegetation heights into a raster (GeoTIFF).

    Parameters:
        vegetation_df (pd.DataFrame): DataFrame containing vegetation points (X, Y, Z_normalized).
        x_min (float): Minimum X coordinate of the grid.
        y_max (float): Maximum Y coordinate of the grid.
        resolution (float): Grid resolution.
        output_file (str): Output filename for GeoTIFF.
    """
    grid_width = int((vegetation_df['X'].max() - x_min) / resolution) + 1
    grid_height = int((y_max - vegetation_df['Y'].min()) / resolution) + 1

    # Initialize empty raster grid (filled with NaNs)
    veg_raster = np.full((grid_height, grid_width), np.nan)

    # Compute row, col indices for each vegetation point
    col_indices = ((vegetation_df['X'] - x_min) / resolution).astype(int)
    row_indices = ((y_max - vegetation_df['Y']) / resolution).astype(int)

    # Assign normalized vegetation height values
    veg_raster[row_indices, col_indices] = vegetation_df['Z_normalized']

    # Define raster transformation
    transform = from_origin(x_min, y_max, resolution, resolution)

    # Save raster as GeoTIFF
    with rasterio.open(
        output_file,
        "w",
        driver="GTiff",
        height=grid_height,
        width=grid_width,
        count=1,
        dtype=vegetation_df['Z_normalized'].dtype,
        crs="EPSG:4326",  # Adjust CRS if necessary
        transform=transform,
        nodata=np.nan
    ) as dst:
        dst.write(veg_raster, 1)

    print(f"✅ Normalized vegetation height raster saved to: {output_file}")

In [ ]:
# Normalize the vegetation points w.r.t. the generated DTM

def normalize_vegetation_with_dtm(vegetation_df, dtm, x_min, y_max, resolution):
    """
    Normalizes vegetation heights using the generated DTM grid.

    Parameters:
        vegetation_df (pd.DataFrame): DataFrame containing vegetation (X, Y, Z).
        dtm (np.array): 2D array representing the DTM grid.
        x_min (float): Minimum X coordinate of the DTM grid.
        y_max (float): Maximum Y coordinate of the DTM grid.
        resolution (float): Grid resolution (e.g., 10m).

    Returns:
        pd.DataFrame: DataFrame with normalized vegetation heights.
    """
    # Get grid dimensions
    dtm_height, dtm_width = dtm.shape

    # Compute row, col indices for each vegetation point
    col_indices = ((vegetation_df['X'] - x_min) / resolution).astype(int)
    row_indices = ((y_max - vegetation_df['Y']) / resolution).astype(int)

    # Ensure indices are within bounds
    row_indices = np.clip(row_indices, 0, dtm_height - 1)
    col_indices = np.clip(col_indices, 0, dtm_width - 1)

    # Extract DTM height values at vegetation locations
    terrain_heights = dtm[row_indices, col_indices]

    # Compute normalized heights (Vegetation Height - Terrain Height)
    vegetation_df['Z_normalized'] = vegetation_df['Z'] - terrain_heights

    return vegetation_df

In [ ]:
# Save normalized vegetation points to a GeoTIFF file - CHM
def save_normalized_vegetation_raster(vegetation_df, x_min, y_max, resolution, output_file):
    """
    Converts normalized vegetation heights into a raster (GeoTIFF).

    Parameters:
        vegetation_df (pd.DataFrame): DataFrame containing vegetation points (X, Y, Z_normalized).
        x_min (float): Minimum X coordinate of the grid.
        y_max (float): Maximum Y coordinate of the grid.
        resolution (float): Grid resolution.
        output_file (str): Output filename for GeoTIFF.
    """
    grid_width = int((vegetation_df['X'].max() - x_min) / resolution) + 1
    grid_height = int((y_max - vegetation_df['Y'].min()) / resolution) + 1

    # Initialize empty raster grid (filled with NaNs)
    veg_raster = np.full((grid_height, grid_width), np.nan)

    # Compute row, col indices for each vegetation point
    col_indices = ((vegetation_df['X'] - x_min) / resolution).astype(int)
    row_indices = ((y_max - vegetation_df['Y']) / resolution).astype(int)

    # Assign normalized vegetation height values
    veg_raster[row_indices, col_indices] = vegetation_df['Z_normalized']

    # Define raster transformation
    transform = from_origin(x_min, y_max, resolution, resolution)

    # Save raster as GeoTIFF
    with rasterio.open(
        output_file,
        "w",
        driver="GTiff",
        height=grid_height,
        width=grid_width,
        count=1,
        dtype=vegetation_df['Z_normalized'].dtype,
        crs="EPSG:4326",  # Adjust CRS if necessary
        transform=transform,
        nodata=np.nan
    ) as dst:
        dst.write(veg_raster, 1)

    print(f"✅ Normalized vegetation height raster saved to: {output_file}")

In [ ]:
# Save normalized vegetation points to ASCII file
def save_normalized_vegetation_ascii(vegetation_df, output_file):
    """
    Saves normalized vegetation points as an ASCII file (space-separated).

    Parameters:
        vegetation_df (pd.DataFrame): DataFrame with X, Y, Z_normalized.
        output_file (str): Output filename.
    """
    vegetation_df[['X', 'Y', 'Z_normalized']].to_csv(output_file, sep=" ", index=False, header=False)
    print(f"✅ Normalized vegetation data saved to: {output_file}")

In [ ]:
# Main Workflow
file_path = "your_lidar_ground_points.txt"  # Change this to your LiDAR points file path
dtm_file_path = "dtm.tif"  # Change this to your previously generated DTM file path
output_veg_raster = "normalized_vegetation.tif"

# Read vegetation points
vegetation_points = read_vegetation_points(file_path)

# Load the DTM
dtm, x_min, y_max, resolution = load_dtm_from_tif(dtm_file_path)

# Normalize vegetation points using the loaded DTM
vegetation_normalized = normalize_vegetation(vegetation_points, dtm, x_min, y_max, resolution)

# Save the normalized vegetation points to a GeoTIFF file
save_normalized_vegetation_raster(vegetation_normalized, x_min, y_max, resolution, output_veg_raster)

# Save the normalized vegetation points as an ASCII file
save_normalized_vegetation_ascii(vegetation_normalized, output_ascii_file)